# Preprocessing Forecast Nilai Pasar Musim Berikutnya

Notebook ini memperlihatkan seluruh proses preprocessing secara bertahap. Data mentah dan scraper hanya dibaca. Fitur pemain pada musim t dipasangkan dengan nilai pasar pada musim t+1.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from unidecode import unidecode
from IPython.display import display

PROJECT_ROOT = Path.cwd()
RAW_PLAYERS_FILE = PROJECT_ROOT / "data" / "raw" / "players_raw.csv"
FBREF_FILE = PROJECT_ROOT / "data" / "interim" / "fbref_player_stats.csv"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "data" / "model"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

CLEAN_FILE = PROCESSED_DIR / "transfermarkt_clean.csv"
FORECAST_DATA_FILE = PROCESSED_DIR / "forecast_dataset.csv"
MODEL_FILE = MODEL_DIR / "players_model.csv"
FEATURE_LIST_FILE = MODEL_DIR / "feature_list.json"
DATASET_SUMMARY_FILE = MODEL_DIR / "dataset_summary.csv"
PREPROCESSING_REPORT_FILE = PROCESSED_DIR / "preprocessing_report.csv"
MATCHING_FILE = INTERIM_DIR / "player_matching_result.csv"
UNMATCHED_FILE = INTERIM_DIR / "unmatched_players.csv"

MIN_MARKET_VALUE = 5.0
TARGET_COLUMN = "next_season_market_value_mio"
LOW_LIMIT = 15.0
HIGH_LIMIT = 35.0

for directory in [PROCESSED_DIR, MODEL_DIR, INTERIM_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw Transfermarkt:", RAW_PLAYERS_FILE)
print("Input FBref:", FBREF_FILE)

Project root: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data
Raw Transfermarkt: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data\data\raw\players_raw.csv
Input FBref: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data\data\interim\fbref_player_stats.csv


## 1. Fungsi Dasar

Fungsi berikut menangani parsing nilai pasar, normalisasi nama, pembagian aman, dan kategori interpretasi 5-15-35.

In [2]:
def require_file(path):
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"File input tidak tersedia: {path}")


def parse_market_value(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).lower().strip().replace("\u00a0", " ")
    if text in {"", "-", "nan", "none"}:
        return np.nan
    text = text.replace("â‚¬", "").replace("€", "").replace("eur", "")
    match = re.search(r"([0-9]+(?:[.,][0-9]+)?)", text)
    if not match:
        return np.nan
    number = float(match.group(1).replace(",", "."))
    if "bn" in text:
        return number * 1000
    if "k" in text or "th" in text:
        return number / 1000
    return number


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = unidecode(str(value)).lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text)).strip()


def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce").replace(0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan).fillna(0)


def value_category(value):
    if pd.isna(value):
        return "Belum tersedia"
    if value < LOW_LIMIT:
        return "Rendah"
    if value <= HIGH_LIMIT:
        return "Menengah"
    return "Tinggi"


print("Fungsi dasar siap.")

Fungsi dasar siap.


## 2. Membaca dan Memeriksa Data Input

In [3]:
require_file(RAW_PLAYERS_FILE)
require_file(FBREF_FILE)

players_raw = pd.read_csv(RAW_PLAYERS_FILE)
fbref_raw = pd.read_csv(FBREF_FILE)

required_columns = {
    "player_id", "player_name", "age", "height_m", "preferred_foot",
    "pos_category", "club", "club_total_mv_raw", "league",
    "league_rank", "season", "market_value_mio",
}
missing_columns = sorted(required_columns - set(players_raw.columns))
if missing_columns:
    raise ValueError(f"Kolom raw tidak lengkap: {missing_columns}")

print(f"Transfermarkt rows: {len(players_raw):,}")
print(f"FBref rows: {len(fbref_raw):,}")
display(players_raw.head())
display(fbref_raw.head())

Transfermarkt rows: 30,024
FBref rows: 22,425


,player_id,player_name,player_url,shirt_number,pos_category,position_detail,age,date_of_birth,nationality,height_m,preferred_foot,club,club_total_mv_raw,league,league_rank,season,market_value_str,market_value_mio
0,238223,Ederson,/ederson/profil/spieler/238223,31,Goalkeeper,NaN,24.0,17/08/1993,Brazil,1.88,left,Manchester City,€1.01bn,Premier League,1,2017,€50.00m,50.0
1,40204,Joe Hart,/joe-hart/profil/spieler/40204,-,Goalkeeper,NaN,31.0,19/04/1987,England,1.96,right,Manchester City,€1.01bn,Premier League,1,2017,€10.00m,10.0
2,40423,Claudio Bravo,/claudio-bravo/profil/spieler/40423,1,Goalkeeper,NaN,35.0,13/04/1983,Chile,1.84,right,Manchester City,€1.01bn,Premier League,1,2017,€3.50m,3.5
3,201574,Angus Gunn,/angus-gunn/profil/spieler/201574,54,Goalkeeper,NaN,22.0,22/01/1996,Scotland,1.96,right,Manchester City,€1.01bn,Premier League,1,2017,€2.00m,2.0
4,186590,John Stones,/john-stones/profil/spieler/186590,5,Defender,NaN,24.0,28/05/1994,England,1.88,right,Manchester City,€1.01bn,Premier League,1,2017,€50.00m,50.0


,league,season,team,player,nation,pos,age,born,standard_playing_time_mp,standard_playing_time_starts,...,keeper_performance_w,keeper_performance_d,keeper_performance_l,keeper_performance_cs,keeper_performance_cspct,keeper_penalty_kicks_pkatt,keeper_penalty_kicks_pka,keeper_penalty_kicks_pksv,keeper_penalty_kicks_pkm,keeper_penalty_kicks_savepct
0,ENG-Premier League,2017,Arsenal,Aaron Ramsey,WAL,MF,26.0,1990.0,24,21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ENG-Premier League,2017,Arsenal,Ainsley Maitland-Niles,ENG,"DF,MF",19.0,1997.0,15,8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ENG-Premier League,2017,Arsenal,Alex Iwobi,NGA,"MF,FW",21.0,1996.0,26,22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENG-Premier League,2017,Arsenal,Alex Oxlade-Chamberlain,ENG,MF,23.0,1993.0,3,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENG-Premier League,2017,Arsenal,Alexandre Lacazette,FRA,FW,26.0,1991.0,32,26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Pembersihan Transfermarkt dan Riwayat Nilai Pasar

Nilai musim berjalan tetap boleh menjadi fitur karena targetnya adalah nilai pada musim berikutnya.

In [4]:
def clean_transfermarkt(frame):
    data = frame.copy()
    raw_rows = len(data)

    for column in ["age", "height_m", "league_rank", "season", "market_value_mio"]:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    data["club_total_mv_mio"] = data["club_total_mv_raw"].map(parse_market_value)

    data = data[data["season"].between(2017, 2024)].copy()
    data = data.dropna(subset=["player_id", "season", "market_value_mio"])
    before_duplicates = len(data)
    data = data.sort_values(["player_id", "season"]).drop_duplicates(
        ["player_id", "season"], keep="last"
    )

    for column in ["player_name", "preferred_foot", "pos_category", "club", "league", "nationality"]:
        if column in data.columns:
            data[column] = data[column].fillna("Unknown").replace("", "Unknown")

    for column in ["age", "height_m", "league_rank", "club_total_mv_mio"]:
        position_median = data.groupby("pos_category")[column].transform("median")
        data[column] = data[column].fillna(position_median).fillna(data[column].median()).fillna(0)

    report = {
        "raw_rows": raw_rows,
        "rows_after_cleaning": len(data),
        "duplicate_rows_removed": before_duplicates - len(data),
    }
    return data, report


players_clean, cleaning_report = clean_transfermarkt(players_raw)

history = players_clean[["player_id", "season", "market_value_mio"]]
previous = history.rename(columns={"market_value_mio": "previous_market_value_mio"}).copy()
previous["season"] += 1
two_ago = history.rename(columns={"market_value_mio": "two_seasons_ago_market_value_mio"}).copy()
two_ago["season"] += 2
future = history.rename(columns={"market_value_mio": TARGET_COLUMN}).copy()
future["season"] -= 1

forecast_data = players_clean.merge(previous, on=["player_id", "season"], how="left")
forecast_data = forecast_data.merge(two_ago, on=["player_id", "season"], how="left")
forecast_data = forecast_data.merge(future, on=["player_id", "season"], how="left")
forecast_data = forecast_data[forecast_data["market_value_mio"] >= MIN_MARKET_VALUE].copy()

print(cleaning_report)
print(f"Rows minimal EUR 5 juta: {len(forecast_data):,}")
display(forecast_data[[
    "player_name", "season", "market_value_mio",
    "previous_market_value_mio", TARGET_COLUMN
]].head(10))

{'raw_rows': 30024, 'rows_after_cleaning': 26741, 'duplicate_rows_removed': 1593}
Rows minimal EUR 5 juta: 10,211


,player_name,season,market_value_mio,previous_market_value_mio,next_season_market_value_mio
32,Wayne Rooney,2017,10.0,NaN,NaN
33,James Milner,2017,15.0,NaN,15.0
34,James Milner,2018,15.0,15.0,6.5
35,James Milner,2019,6.5,15.0,3.0
69,Arjen Robben,2017,7.0,NaN,4.0
123,Mario Gómez,2017,6.0,NaN,2.0
129,Christian Fuchs,2017,5.0,NaN,3.0
148,Jonas Hofmann,2018,9.0,3.0,7.0
149,Jonas Hofmann,2019,7.0,9.0,16.0
150,Jonas Hofmann,2020,16.0,7.0,13.0


## 4. Menyiapkan Statistik FBref

In [5]:
PERFORMANCE_FEATURES = [
    "matches_played", "starts", "minutes", "goals", "assists",
    "yellow_cards", "red_cards", "shots_total", "shots_on_target",
    "fouls_committed", "fouls_drawn", "saves", "clean_sheets",
    "goals_against", "shots_on_target_against", "goals_per_90",
    "assists_per_90", "goal_assist_per_90", "shots_per_90",
    "shots_on_target_per_90", "cards_per_90", "starts_rate",
    "save_pct", "clean_sheet_pct", "has_performance_stats",
]


def first_numeric(frame, columns):
    result = pd.Series(np.nan, index=frame.index, dtype="float64")
    for column in columns:
        if column in frame.columns:
            result = result.fillna(pd.to_numeric(frame[column], errors="coerce"))
    return result.fillna(0)


def prepare_fbref(frame):
    fb = frame.copy()
    fb["season"] = pd.to_numeric(fb["season"], errors="coerce")
    fb = fb[fb["season"].between(2017, 2024)].copy()
    fb["player_key"] = fb["player"].map(normalize_text)
    fb["club_key"] = fb["team"].map(normalize_text)

    out = fb[["season", "player", "team", "player_key", "club_key"]].copy()
    mapping = {
        "matches_played": ["standard_playing_time_mp", "keeper_playing_time_mp"],
        "starts": ["standard_playing_time_starts", "keeper_playing_time_starts"],
        "minutes": ["standard_playing_time_min", "keeper_playing_time_min"],
        "goals": ["standard_performance_gls", "shooting_standard_gls"],
        "assists": ["standard_performance_ast"],
        "yellow_cards": ["standard_performance_crdy", "misc_performance_crdy"],
        "red_cards": ["standard_performance_crdr", "misc_performance_crdr"],
        "shots_total": ["shooting_standard_sh"],
        "shots_on_target": ["shooting_standard_sot"],
        "fouls_committed": ["misc_performance_fls"],
        "fouls_drawn": ["misc_performance_fld"],
        "saves": ["keeper_performance_saves"],
        "clean_sheets": ["keeper_performance_cs"],
        "goals_against": ["keeper_performance_ga"],
        "shots_on_target_against": ["keeper_performance_sota"],
    }
    for output_column, source_columns in mapping.items():
        out[output_column] = first_numeric(fb, source_columns)

    nineties = out["minutes"] / 90
    out["goals_per_90"] = safe_divide(out["goals"], nineties)
    out["assists_per_90"] = safe_divide(out["assists"], nineties)
    out["goal_assist_per_90"] = safe_divide(out["goals"] + out["assists"], nineties)
    out["shots_per_90"] = safe_divide(out["shots_total"], nineties)
    out["shots_on_target_per_90"] = safe_divide(out["shots_on_target"], nineties)
    out["cards_per_90"] = safe_divide(out["yellow_cards"] + out["red_cards"], nineties)
    out["starts_rate"] = safe_divide(out["starts"], out["matches_played"])
    out["save_pct"] = safe_divide(out["saves"], out["shots_on_target_against"])
    out["clean_sheet_pct"] = safe_divide(out["clean_sheets"], out["matches_played"])
    out["has_performance_stats"] = 1
    return out


fbref_features = prepare_fbref(fbref_raw)
print(f"FBref features prepared: {len(fbref_features):,}")
display(fbref_features.head())

FBref features prepared: 22,425


,season,player,team,player_key,club_key,matches_played,starts,minutes,goals,assists,...,goals_per_90,assists_per_90,goal_assist_per_90,shots_per_90,shots_on_target_per_90,cards_per_90,starts_rate,save_pct,clean_sheet_pct,has_performance_stats
0,2017,Aaron Ramsey,Arsenal,aaron ramsey,arsenal,24.0,21.0,1846.0,7.0,8.0,...,0.341278,0.390033,0.731311,2.730228,1.218852,0.000000,0.875000,0.0,0.0,1
1,2017,Ainsley Maitland-Niles,Arsenal,ainsley maitland niles,arsenal,15.0,8.0,914.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.492341,0.000000,0.098468,0.533333,0.0,0.0,1
2,2017,Alex Iwobi,Arsenal,alex iwobi,arsenal,26.0,22.0,1830.0,3.0,5.0,...,0.147541,0.245902,0.393443,2.213115,1.131148,0.049180,0.846154,0.0,0.0,1
3,2017,Alex Oxlade-Chamberlain,Arsenal,alex oxlade chamberlain,arsenal,3.0,3.0,241.0,0.0,0.0,...,0.000000,0.000000,0.000000,3.360996,0.746888,0.000000,1.000000,0.0,0.0,1
4,2017,Alexandre Lacazette,Arsenal,alexandre lacazette,arsenal,32.0,26.0,2202.0,14.0,4.0,...,0.572207,0.163488,0.735695,2.779292,1.430518,0.040872,0.812500,0.0,0.0,1


## 5. Pencocokan Transfermarkt dan FBref

Pencocokan dilakukan pada musim yang sama. Nama eksak diprioritaskan, kemudian fuzzy matching dengan pemeriksaan klub.

In [6]:
def match_fbref(players, fbref):
    result = players.copy().reset_index(drop=True)
    for column in PERFORMANCE_FEATURES:
        result[column] = 0.0

    by_season = {}
    keys_by_season = {}
    for season, group in fbref.groupby("season"):
        mapping = {}
        for record in group.to_dict("records"):
            mapping.setdefault(record["player_key"], []).append(record)
        by_season[int(season)] = mapping
        keys_by_season[int(season)] = list(mapping)

    audits = []
    for index, row in result.iterrows():
        season = int(row["season"])
        name_key = normalize_text(row["player_name"])
        club_key = normalize_text(row["club"])
        mapping = by_season.get(season, {})
        candidates = mapping.get(name_key, [])
        match_type = "exact"
        name_score = 100

        if not candidates and mapping:
            fuzzy = process.extractOne(
                name_key, keys_by_season[season],
                scorer=fuzz.WRatio, score_cutoff=90
            )
            if fuzzy:
                candidates = mapping[fuzzy[0]]
                name_score = int(fuzzy[1])
                match_type = "fuzzy"

        chosen = None
        club_score = 0
        if candidates:
            chosen = max(
                candidates,
                key=lambda item: fuzz.token_sort_ratio(club_key, item["club_key"])
            )
            club_score = int(fuzz.token_sort_ratio(club_key, chosen["club_key"]))
            if match_type == "fuzzy" and club_score < 65:
                chosen = None

        matched = chosen is not None
        if matched:
            for column in PERFORMANCE_FEATURES:
                result.at[index, column] = chosen[column]

        audits.append({
            "row_id": index,
            "player_id": row["player_id"],
            "player_name": row["player_name"],
            "club": row["club"],
            "season": season,
            "matched": matched,
            "match_type": match_type if matched else "none",
            "name_score": name_score if matched else 0,
            "club_score": club_score if matched else 0,
            "fbref_player_name": chosen["player"] if matched else "",
            "fbref_club": chosen["team"] if matched else "",
        })

    return result, pd.DataFrame(audits)


forecast_data, matching_audit = match_fbref(forecast_data, fbref_features)
matched_rows = int(matching_audit["matched"].sum())
match_rate = matched_rows / len(matching_audit)

print(f"Matched: {matched_rows:,} dari {len(matching_audit):,} ({match_rate:.2%})")
display(matching_audit["match_type"].value_counts().rename_axis("match_type").reset_index(name="rows"))
display(matching_audit.head())

Matched: 9,793 dari 10,211 (95.91%)


,match_type,rows
0,exact,9578
1,none,418
2,fuzzy,215


,row_id,player_id,player_name,club,season,matched,match_type,name_score,club_score,fbref_player_name,fbref_club
0,0,3332,Wayne Rooney,Everton FC,2017,True,exact,100,82,Wayne Rooney,Everton
1,1,3333,James Milner,Liverpool FC,2017,True,exact,100,85,James Milner,Liverpool
2,2,3333,James Milner,Liverpool FC,2018,True,exact,100,85,James Milner,Liverpool
3,3,3333,James Milner,Liverpool FC,2019,True,exact,100,85,James Milner,Liverpool
4,4,4360,Arjen Robben,Bayern Munich,2017,True,exact,100,100,Arjen Robben,Bayern Munich


## 6. Feature Engineering dan Target Forecast

In [7]:
forecast_data["target_season"] = forecast_data["season"] + 1
forecast_data["current_market_value_mio"] = forecast_data["market_value_mio"]
forecast_data["current_market_value_log"] = np.log1p(forecast_data["current_market_value_mio"])
forecast_data["current_market_value_category"] = forecast_data["current_market_value_mio"].map(value_category)
forecast_data["next_season_market_value_category"] = forecast_data[TARGET_COLUMN].map(value_category)
forecast_data["has_previous_market_value"] = forecast_data["previous_market_value_mio"].notna().astype(int)
forecast_data = forecast_data.sort_values(["player_id", "season"]).copy()
forecast_data["market_value_history_count"] = forecast_data.groupby("player_id").cumcount()
forecast_data["historical_growth_rate"] = safe_divide(
    forecast_data["current_market_value_mio"] - forecast_data["previous_market_value_mio"],
    forecast_data["previous_market_value_mio"],
).clip(-5, 5)
forecast_data["previous_market_value_mio"] = forecast_data["previous_market_value_mio"].fillna(0)
forecast_data["two_seasons_ago_market_value_mio"] = forecast_data["two_seasons_ago_market_value_mio"].fillna(0)

league_average = forecast_data.groupby(["league", "season"])["club_total_mv_mio"].transform("mean")
forecast_data["club_mv_relative_to_league_avg"] = safe_divide(
    forecast_data["club_total_mv_mio"], league_average
)
forecast_data["current_value_to_club_ratio"] = safe_divide(
    forecast_data["current_market_value_mio"], forecast_data["club_total_mv_mio"]
)
forecast_data["has_known_next_season_value"] = forecast_data[TARGET_COLUMN].notna().astype(int)
forecast_data["value_change_next_season_mio"] = (
    forecast_data[TARGET_COLUMN] - forecast_data["current_market_value_mio"]
)

MODEL_FEATURES = [
    "age", "height_m", "preferred_foot", "pos_category", "league",
    "league_rank", "club_total_mv_mio", "club_mv_relative_to_league_avg",
    "current_market_value_mio", "current_market_value_log",
    "current_market_value_category", "previous_market_value_mio",
    "two_seasons_ago_market_value_mio", "has_previous_market_value",
    "market_value_history_count", "historical_growth_rate",
    "current_value_to_club_ratio", "minutes", "goals", "assists",
    "yellow_cards", "shots_total", "shots_on_target", "starts_rate",
    "goals_per_90", "assists_per_90", "goal_assist_per_90",
    "shots_per_90", "has_performance_stats",
]

model_data = forecast_data[
    forecast_data[TARGET_COLUMN].notna() & (forecast_data[TARGET_COLUMN] > 0)
].copy()
model_columns = [
    "player_id", "player_name", "club", "season", "target_season",
    *MODEL_FEATURES, TARGET_COLUMN, "next_season_market_value_category",
]
model_data = model_data[model_columns]

print(f"Forecast dataset rows: {len(forecast_data):,}")
print(f"Model labeled rows: {len(model_data):,}")
print(f"Features: {len(MODEL_FEATURES)}")
display(model_data.head())

Forecast dataset rows: 10,211
Model labeled rows: 8,090
Features: 29


,player_id,player_name,club,season,target_season,age,height_m,preferred_foot,pos_category,league,...,shots_total,shots_on_target,starts_rate,goals_per_90,assists_per_90,goal_assist_per_90,shots_per_90,has_performance_stats,next_season_market_value_mio,next_season_market_value_category
1,3333,James Milner,Liverpool FC,2017,2018,32.0,1.75,right,Midfield,Premier League,...,21.0,5.0,0.500000,0.000000,0.152113,0.152113,1.064789,1.0,15.0,Menengah
2,3333,James Milner,Liverpool FC,2018,2019,33.0,1.75,right,Midfield,Premier League,...,24.0,8.0,0.612903,0.251537,0.201230,0.452767,1.207378,1.0,6.5,Rendah
3,3333,James Milner,Liverpool FC,2019,2020,34.0,1.75,right,Midfield,Premier League,...,12.0,4.0,0.409091,0.192102,0.192102,0.384205,1.152615,1.0,3.0,Rendah
4,4360,Arjen Robben,Bayern Munich,2017,2018,34.0,1.80,left,Attack,Bundesliga,...,51.0,21.0,0.904762,0.298607,0.298607,0.597213,3.045786,1.0,4.0,Rendah
5,6288,Mario Gómez,VfB Stuttgart,2017,2018,32.0,1.89,right,Attack,Bundesliga,...,33.0,15.0,1.000000,0.519856,0.000000,0.519856,2.144404,1.0,2.0,Rendah


## 7. Validasi Anti Target Leakage dan Distribusi Split

In [8]:
forbidden_features = [feature for feature in MODEL_FEATURES if feature.startswith("next_season_")]
if forbidden_features:
    raise ValueError(f"Target leakage ditemukan: {forbidden_features}")
if not (model_data["target_season"] == model_data["season"] + 1).all():
    raise ValueError("Pasangan feature season dan target season tidak konsisten.")
if model_data.duplicated(["player_id", "season"]).any():
    raise ValueError("Duplikasi player-season ditemukan.")

split_summary = pd.DataFrame([
    {"split": "Train", "target_season": "2018-2022", "rows": int((model_data["target_season"] <= 2022).sum())},
    {"split": "Validation", "target_season": "2023", "rows": int((model_data["target_season"] == 2023).sum())},
    {"split": "Test", "target_season": "2024", "rows": int((model_data["target_season"] == 2024).sum())},
    {"split": "Forecast", "target_season": "2025", "rows": int((forecast_data["season"] == 2024).sum())},
])
display(split_summary)
display(
    model_data.groupby(["target_season", "next_season_market_value_category"])
    .size().unstack(fill_value=0)
)

,split,target_season,rows
0,Train,2018-2022,5701
1,Validation,2023,1190
2,Test,2024,1199
3,Forecast,2025,1397


next_season_market_value_category,Menengah,Rendah,Tinggi
target_season,,,
2018,353,536,164
2019,317,743,136
2020,364,580,152
2021,385,662,146
2022,368,650,145
2023,375,649,166
2024,415,609,175


## 8. Menyimpan Hasil Preprocessing

Jalankan sel ini setelah seluruh validasi berhasil.

In [9]:
forecast_data = forecast_data.sort_values(
    ["season", "league", "club", "player_name"]
).reset_index(drop=True)

forecast_data.to_csv(CLEAN_FILE, index=False)
forecast_data.to_csv(FORECAST_DATA_FILE, index=False)
model_data.to_csv(MODEL_FILE, index=False)
matching_audit.to_csv(MATCHING_FILE, index=False)
matching_audit[~matching_audit["matched"]].to_csv(UNMATCHED_FILE, index=False)

with open(FEATURE_LIST_FILE, "w", encoding="utf-8") as file:
    json.dump(MODEL_FEATURES, file, indent=2)

dataset_summary = pd.DataFrame([
    {"metric": "raw_rows_read_only", "value": len(players_raw)},
    {"metric": "forecast_rows", "value": len(forecast_data)},
    {"metric": "labeled_model_rows", "value": len(model_data)},
    {"metric": "forecast_2025_candidates", "value": int((forecast_data["season"] == 2024).sum())},
    {"metric": "features", "value": len(MODEL_FEATURES)},
    {"metric": "train_rows", "value": int((model_data["target_season"] <= 2022).sum())},
    {"metric": "validation_rows", "value": int((model_data["target_season"] == 2023).sum())},
    {"metric": "test_rows", "value": int((model_data["target_season"] == 2024).sum())},
    {"metric": "fbref_matched_rows", "value": matched_rows},
])
dataset_summary.to_csv(DATASET_SUMMARY_FILE, index=False)
dataset_summary.to_csv(PREPROCESSING_REPORT_FILE, index=False)

print("Preprocessing selesai.")
display(dataset_summary)
print("Model dataset:", MODEL_FILE)
print("Forecast dataset:", FORECAST_DATA_FILE)

Preprocessing selesai.


,metric,value
0,raw_rows_read_only,30024
1,forecast_rows,10211
2,labeled_model_rows,8090
3,forecast_2025_candidates,1397
4,features,29
5,train_rows,5701
6,validation_rows,1190
7,test_rows,1199
8,fbref_matched_rows,9793


Model dataset: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data\data\model\players_model.csv
Forecast dataset: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data\data\processed\forecast_dataset.csv
